# Sesión 1: Demo - Probabilidades en Language Models

**Curso:** Introducción a Large Language Models  
**Profesor:** Francisco Pérez Carbajal  
**Semana:** 1 (Fundamentos)  

---

## Objetivo

Entender cómo un modelo de lenguaje asigna **probabilidades** a secuencias de texto.

**Concepto clave:** Un language model es una distribución de probabilidad sobre secuencias de tokens:

$$p(x_1, x_2, \ldots, x_L)$$

Usando la **chain rule**, podemos descomponerla:

$$p(x_1, x_2, \ldots, x_L) = p(x_1) \cdot p(x_2 | x_1) \cdot p(x_3 | x_1, x_2) \cdots p(x_L | x_1, \ldots, x_{L-1})$$

## 1. Setup

Instalar las bibliotecas necesarias:

In [ ]:
!pip install transformers torch -q

In [ ]:
import torch
import torch.nn.functional as F
from transformers import GPT2LMHeadModel, GPT2Tokenizer
import pandas as pd

print(f"✓ Transformers version: {transformers.__version__}")
print(f"✓ PyTorch version: {torch.__version__}")

## 2. Cargar un modelo pre-entrenado

Usaremos **GPT-2**, un modelo autoregresivo que predice el siguiente token dado el contexto.

In [ ]:
# Cargar modelo y tokenizer
model_name = 'gpt2'  # ~124M parámetros

print(f"Cargando {model_name}...")
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)

# Poner en modo evaluación (no vamos a entrenar)
model.eval()

print("✓ Modelo cargado correctamente")

## 3. Tokenización: De texto a números

Los LLMs no trabajan directamente con texto, sino con **tokens** (representados como números).

In [ ]:
# Ejemplo de tokenización
text = "The cat sat on the"

# Ver los tokens como texto
tokens_text = tokenizer.tokenize(text)
print(f"Texto original: '{text}'")
print(f"Tokens: {tokens_text}")

# Ver los IDs numéricos
token_ids = tokenizer.encode(text)
print(f"Token IDs: {token_ids}")

# Decodificar de vuelta
decoded = tokenizer.decode(token_ids)
print(f"Decodificado: '{decoded}'")

## 4. Calcular probabilidades del siguiente token

Dado un contexto, ¿qué tan probable es cada posible siguiente token?

In [ ]:
def get_next_token_probabilities(text, top_k=10):
    """
    Dado un texto, retorna las probabilidades de los top-k siguientes tokens.
    
    Args:
        text (str): Contexto de texto
        top_k (int): Número de tokens más probables a mostrar
    
    Returns:
        pandas.DataFrame: Tabla con tokens y sus probabilidades
    """
    # Tokenizar
    inputs = tokenizer(text, return_tensors='pt')
    
    # Forward pass (sin calcular gradientes)
    with torch.no_grad():
        outputs = model(**inputs)
    
    # Obtener logits del último token
    next_token_logits = outputs.logits[0, -1, :]
    
    # Convertir a probabilidades (softmax)
    next_token_probs = F.softmax(next_token_logits, dim=-1)
    
    # Obtener top-k tokens más probables
    top_probs, top_indices = torch.topk(next_token_probs, top_k)
    
    # Crear tabla
    results = []
    for prob, idx in zip(top_probs.tolist(), top_indices.tolist()):
        token = tokenizer.decode([idx])
        results.append({
            'Token': repr(token),  # repr para ver espacios
            'Probabilidad': f'{prob:.4f}',
            'Porcentaje': f'{prob*100:.2f}%'
        })
    
    return pd.DataFrame(results)

### Experimento 1: "The cat sat on the"

¿Qué viene después de "the"?

In [ ]:
text = "The cat sat on the"
print(f"Contexto: '{text}'")
print("\nTop 10 siguientes tokens más probables:\n")

df = get_next_token_probabilities(text, top_k=10)
display(df)

**Pregunta para discusión:**
- ¿Por qué " mat" (alfombra) es más probable que " dog" (perro)?
- ¿Qué tipo de "conocimiento" necesita el modelo para hacer esto?

### Experimento 2: Cambiar el sujeto

¿Qué pasa si cambiamos "cat" por "dog"?

In [ ]:
text = "The dog sat on the"
print(f"Contexto: '{text}'")
print("\nTop 10 siguientes tokens más probables:\n")

df = get_next_token_probabilities(text, top_k=10)
display(df)

**Observación:** Las probabilidades cambian dependiendo del contexto completo.

## 5. Experimentación libre

Ahora es tu turno. Prueba diferentes contextos y observa las probabilidades:

In [ ]:
# Prueba tus propios contextos aquí
text = "To be or not to"  # Cambia esto

print(f"Contexto: '{text}'")
print("\nTop 10 siguientes tokens más probables:\n")

df = get_next_token_probabilities(text, top_k=10)
display(df)

### Ideas para probar:

1. **Frases famosas:**
   - "To be or not to"
   - "Once upon a"
   - "In the beginning"

2. **Contextos técnicos:**
   - "The Python code to calculate"
   - "The derivative of x squared is"

3. **Preguntas:**
   - "The capital of France is"
   - "Water boils at"

## 6. Comparar dos continuaciones

¿Qué secuencia es más probable según el modelo?

In [ ]:
def compare_sequences(seq1, seq2):
    """
    Compara las probabilidades de dos secuencias.
    """
    results = []
    
    for seq in [seq1, seq2]:
        # Tokenizar
        inputs = tokenizer(seq, return_tensors='pt')
        
        # Calcular loss (negative log-likelihood)
        with torch.no_grad():
            outputs = model(**inputs, labels=inputs['input_ids'])
            loss = outputs.loss.item()
        
        # Log-probabilidad = -loss
        log_prob = -loss * len(inputs['input_ids'][0])
        
        results.append({
            'Secuencia': seq,
            'Log-probabilidad': f'{log_prob:.2f}',
            'Perplejidad': f'{torch.exp(outputs.loss).item():.2f}'
        })
    
    df = pd.DataFrame(results)
    return df

In [ ]:
# Comparar dos oraciones
seq1 = "The cat sat on the mat"
seq2 = "The mat sat on the cat"

print("Comparación de probabilidades:\n")
df = compare_sequences(seq1, seq2)
display(df)

print("\n📊 Mayor log-probabilidad = más probable")
print("📊 Menor perplejidad = más probable")

## 7. Visualización: Distribución de probabilidades

Veamos gráficamente la distribución de probabilidades del siguiente token:

In [ ]:
import matplotlib.pyplot as plt

def plot_token_probabilities(text, top_k=15):
    """
    Graficar las probabilidades de los top-k tokens.
    """
    # Obtener probabilidades
    inputs = tokenizer(text, return_tensors='pt')
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    next_token_logits = outputs.logits[0, -1, :]
    next_token_probs = F.softmax(next_token_logits, dim=-1)
    
    top_probs, top_indices = torch.topk(next_token_probs, top_k)
    
    # Preparar datos para graficar
    tokens = [tokenizer.decode([idx]) for idx in top_indices.tolist()]
    probs = top_probs.tolist()
    
    # Graficar
    plt.figure(figsize=(12, 6))
    plt.barh(range(len(tokens)), probs)
    plt.yticks(range(len(tokens)), tokens)
    plt.xlabel('Probabilidad')
    plt.title(f'Top-{top_k} tokens más probables después de: "{text}"')
    plt.gca().invert_yaxis()  # Mayor probabilidad arriba
    plt.tight_layout()
    plt.show()

In [ ]:
text = "The cat sat on the"
plot_token_probabilities(text, top_k=15)

## 8. Conclusiones

En esta sesión aprendimos:

1. ✅ Un **language model** es una distribución de probabilidad sobre secuencias
2. ✅ Los modelos **autoregresivos** predicen un token a la vez: $p(x_i | x_1, \ldots, x_{i-1})$
3. ✅ Las probabilidades reflejan **conocimiento implícito**:
   - Sintaxis (gramática)
   - Semántica (significado)
   - Conocimiento del mundo
4. ✅ Podemos **inspeccionar** estas probabilidades con código

---

### Para reflexionar:

- ¿Cómo aprendió el modelo estas probabilidades?
- ¿Qué datos necesitó?
- ¿Qué limitaciones tiene este enfoque?

**Próxima sesión:** Aprenderemos a **generar** texto usando estos modelos.

---

## Ejercicio para casa (opcional)

Experimenta con contextos en **español**. ¿Qué tan bien funciona GPT-2 (que fue entrenado principalmente en inglés)?

Ejemplo:
```python
text = "El gato se sentó en"
df = get_next_token_probabilities(text, top_k=10)
display(df)
```

---

## 📚 Recursos adicionales

**Documentación**
- [Hugging Face docs](https://huggingface.co/docs)

**Lecturas:**
- [CS224N notes on LMs](https://web.stanford.edu/class/cs224n/readings/cs224n-2019-notes05-LM_RNN.pdf)
- [Stanford CS324 - Introduction](https://stanford-cs324.github.io/winter2022/lectures/introduction/)
- [The Illustrated Transformer](https://jalammar.github.io/illustrated-transformer/)

**Interactivos:**
- [Hugging Face Model Hub](https://huggingface.co/models)
- [Transformer Explainer](https://poloclub.github.io/transformer-explainer/)
- [LLM Visualization](http://jalammar.github.io/illustrated-gpt2/)
- [Foro](https://discuss.huggingface.co)

**Cursos:**
- [Hugging Face NLP Course](https://huggingface.co/learn/nlp-course/)
- [Stanford CS224N](http://web.stanford.edu/class/cs224n/)
- [Fast.ai NLP](https://www.fast.ai/)
- [Stanford CS324 full course](https://stanford-cs324.github.io/winter2022/)